# Loan_approved Status - Model Building using KNN

# **Classification**

In [94]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [95]:
df = pd.read_csv(r"C:\Users\ksowm\Downloads\loan_approved.csv")
df

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status (Approved)
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y


In [96]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Loan_ID                 614 non-null    object 
 1   Gender                  601 non-null    object 
 2   Married                 611 non-null    object 
 3   Dependents              599 non-null    object 
 4   Education               614 non-null    object 
 5   Self_Employed           582 non-null    object 
 6   ApplicantIncome         614 non-null    int64  
 7   CoapplicantIncome       614 non-null    float64
 8   LoanAmount              592 non-null    float64
 9   Loan_Amount_Term        600 non-null    float64
 10  Credit_History          564 non-null    float64
 11  Property_Area           614 non-null    object 
 12  Loan_Status (Approved)  614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [97]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


In [98]:
df.shape

(614, 13)

In [99]:
df.rename(columns = {"Loan_Status (Approved)":"Loan_Status_Approved"}, inplace = True)

In [100]:
# Loan_Status is not depends on Loan_ID and dependents , so drop those columns
df.drop(['Loan_ID'],axis = 1,inplace = True)

In [101]:
df.shape

(614, 12)

# Basic Check

In [102]:
df.isnull().sum()

Gender                  13
Married                  3
Dependents              15
Education                0
Self_Employed           32
ApplicantIncome          0
CoapplicantIncome        0
LoanAmount              22
Loan_Amount_Term        14
Credit_History          50
Property_Area            0
Loan_Status_Approved     0
dtype: int64

In [103]:
df.duplicated().sum()

0

In [104]:
df.columns

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area',
       'Loan_Status_Approved'],
      dtype='object')

# Outliers Detecting

In [105]:
num_cols = df.select_dtypes(include = ["int64","float64"]).columns
cat_cols = df.select_dtypes(exclude = ["int64","float64"]).columns

In [106]:
num_cols

Index(['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History'],
      dtype='object')

In [107]:
cat_cols

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'Property_Area', 'Loan_Status_Approved'],
      dtype='object')

In [108]:
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    print(col)
    print("IQR:",IQR)
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR
    print(f"Lower_limit:{lower_limit}")
    print(f"Upper_limit:{upper_limit}")
    print("MAX:",df[col].max())
    print("MIN:",df[col].min())

    outliers = df[(df[col] < lower_limit) | (df[col] > upper_limit)]
    print(f"{col} outliers count:", outliers.shape[0])
    print("-"*40)

ApplicantIncome
IQR: 2917.5
Lower_limit:-1498.75
Upper_limit:10171.25
MAX: 81000
MIN: 150
ApplicantIncome outliers count: 50
----------------------------------------
CoapplicantIncome
IQR: 2297.25
Lower_limit:-3445.875
Upper_limit:5743.125
MAX: 41667.0
MIN: 0.0
CoapplicantIncome outliers count: 18
----------------------------------------
LoanAmount
IQR: 68.0
Lower_limit:-2.0
Upper_limit:270.0
MAX: 700.0
MIN: 9.0
LoanAmount outliers count: 39
----------------------------------------
Loan_Amount_Term
IQR: 0.0
Lower_limit:360.0
Upper_limit:360.0
MAX: 480.0
MIN: 12.0
Loan_Amount_Term outliers count: 88
----------------------------------------
Credit_History
IQR: 0.0
Lower_limit:1.0
Upper_limit:1.0
MAX: 1.0
MIN: 0.0
Credit_History outliers count: 89
----------------------------------------


# Selection of Target and predictors

In [109]:
y = df["Loan_Status_Approved"]

In [110]:
X = df.drop(["Loan_Status_Approved"],axis = 1)

In [111]:
X.columns

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area'],
      dtype='object')

# Train - Test Split 

In [112]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2) 
print(X_train.shape)
print(X_test.shape)

(491, 11)
(123, 11)


# Handling Null values 

In [115]:
cat_cols = X_train.select_dtypes(exclude=["int64","float64"]).columns
num_cols = X_train.select_dtypes(include=["int64","float64"]).columns

- **Fill Categorical Columns (Use Mode)**
- **Fill Numerical Columns (Use Median)**

In [116]:
# Fill categorical nulls
for col in cat_cols:
    mode_val = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(mode_val)
    X_test[col] = X_test[col].fillna(mode_val)

# Fill numerical nulls
for col in num_cols:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

In [118]:
X_train.isnull().sum()

Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
dtype: int64

# Handling Outliers

In [119]:
num_cols = X_train.select_dtypes(include = ["int64","float64"]).columns
cat_cols = X_train.select_dtypes(exclude = ["int64","float64"]).columns

In [120]:
iqr_limits = {}

for col in num_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    iqr_limits[col] = (lower, upper)
    
    lower, upper = iqr_limits[col]
    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col]  = X_test[col].clip(lower, upper)

    
    outliers = X_train[(X_train[col] < lower) | (X_train[col] > upper)]
    print(f"{col} outliers count:", outliers.shape[0])


ApplicantIncome outliers count: 0
CoapplicantIncome outliers count: 0
LoanAmount outliers count: 0
Loan_Amount_Term outliers count: 0
Credit_History outliers count: 0


In [121]:
X_train.isnull().sum()

Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
dtype: int64

In [122]:
print("num_cols:" , num_cols)
print("cat_cols:" , cat_cols)

num_cols: Index(['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History'],
      dtype='object')
cat_cols: Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'Property_Area'],
      dtype='object')


In [123]:
print(X_test.select_dtypes(include="object").columns)

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'Property_Area'],
      dtype='object')


# Apply Ordinal Encoding (Only on Categorical Columns)

In [124]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()

X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

# Apply Standard Scaling (Only on Numerical Columns)

In [125]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [126]:
from sklearn.neighbors import KNeighborsClassifier
kn = KNeighborsClassifier()

In [127]:
kn.fit(X_train,y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [128]:
print(X_test.dtypes)

Gender               float64
Married              float64
Dependents           float64
Education            float64
Self_Employed        float64
ApplicantIncome      float64
CoapplicantIncome    float64
LoanAmount           float64
Loan_Amount_Term     float64
Credit_History       float64
Property_Area        float64
dtype: object


In [129]:
y_pred = kn.predict(X_test)

In [132]:
from sklearn.metrics import  accuracy_score
acc = accuracy_score(y_test,y_pred)
print(acc)

0.5772357723577236


In [133]:
y_train_pred = kn.predict(X_train)

In [134]:
acc = accuracy_score(y_train,y_train_pred)
print(acc)

0.7556008146639511
